# 4 — Calibración de Probabilidades e Incertidumbre

Los clasificadores devuelven `predict_proba` pero no garantizan que las probabilidades estén bien calibradas. Un modelo que dice "90% de confianza" debería acertar el 90% de esas veces.

**Qué se hace aquí:**
1. Evaluar la calibración original de RF, GB y SVM mediante **Reliability Diagrams** y **ECE** (Expected Calibration Error)
2. Aplicar `CalibratedClassifierCV` con métodos **isotónico** y **Platt (sigmoid)**
3. Comparar ECE antes/después
4. Analizar la **entropía de predicción** como proxy de incertidumbre: ¿en qué temperaturas es el modelo más incierto?

> Para multiclase se usa la estrategia one-vs-rest y se promedian los diagramas de fiabilidad.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import accuracy_score, log_loss

CSV_PATH  = Path("dataset_features_temperatura.csv")
FEAT_JSON = Path("features_seleccionadas.json")

In [ ]:
df = pd.read_csv(CSV_PATH)

META_COLS = {
    "sample_uid","sample_id","tx_path","rx_path",
    "tx_path_rel_to_bbdd_parent","rx_path_rel_to_bbdd_parent",
    "split","temperature_folder","N_subcarriers","M_symbols","num_valid","valid_ratio",
}
CONSTANT_COLS = {"phase_jump_m_count","phase_jump_m_ratio","phase_jump_k_count","phase_jump_k_ratio"}

if FEAT_JSON.exists():
    with open(FEAT_JSON) as f: FEAT_COLS = json.load(f)
else:
    FEAT_COLS = [c for c in df.columns if c not in META_COLS and c != "temperature" and c not in CONSTANT_COLS]
print(f"Features: {len(FEAT_COLS)}")

df["label"] = df["temperature"].astype(int).astype(str) + "C"
classes = sorted(df["label"].unique())

def get_split(split):
    d = df[df["split"] == split]
    return d[FEAT_COLS].values, d["label"].values

X_tr, y_tr = get_split("train")
X_v,  y_v  = get_split("val")
X_te, y_te = get_split("test")
# Calibración necesita val como conjunto de calibrado
X_tv = np.vstack([X_tr, X_v])
y_tv = np.concatenate([y_tr, y_v])
print(f"Train+Val: {X_tv.shape}  Test: {X_te.shape}  Clases: {len(classes)}")

In [ ]:
def ece(y_true, proba, classes, n_bins=10):
    """Expected Calibration Error (macro-average OvR)."""
    y_bin = label_binarize(y_true, classes=classes)
    ece_vals = []
    for c in range(proba.shape[1]):
        p = proba[:, c]
        y = y_bin[:, c]
        bins = np.linspace(0, 1, n_bins+1)
        ece_c = 0.0
        for lo, hi in zip(bins[:-1], bins[1:]):
            mask = (p >= lo) & (p < hi)
            if mask.sum() == 0: continue
            ece_c += mask.sum() / len(y) * abs(y[mask].mean() - p[mask].mean())
        ece_vals.append(ece_c)
    return np.mean(ece_vals)

print("ECE helper OK")

In [ ]:
# ── Entrenar modelos base ─────────────────────────────────────────────────────
models_base = {
    "Random Forest": Pipeline([
        ("imp", SimpleImputer(strategy="mean")),
        ("m",   RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ("imp", SimpleImputer(strategy="mean")),
        ("m",   GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42))
    ]),
    "SVM (RBF)": Pipeline([
        ("imp", SimpleImputer(strategy="mean")),
        ("sc",  StandardScaler()),
        ("m",   SVC(kernel="rbf", C=10, probability=True, random_state=42))
    ]),
}

results = {}
for name, pipe in models_base.items():
    print(f"Entrenando {name}...", end=" ", flush=True)
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)
    preds = pipe.classes_[np.argmax(proba, axis=1)]
    acc   = accuracy_score(y_te, preds)
    ll    = log_loss(y_te, proba, labels=pipe.classes_)
    ece_v = ece(y_te, proba, list(pipe.classes_))
    results[name] = {"proba": proba, "acc": acc, "ll": ll, "ece": ece_v,
                     "classes": list(pipe.classes_)}
    print(f"Acc={acc*100:.2f}%  ECE={ece_v:.4f}  LogLoss={ll:.4f}")

In [ ]:
# ── Calibración con isotónico y sigmoid ───────────────────────────────────────
calib_results = {}
for name, pipe in models_base.items():
    calib_results[name] = {}
    for method in ["isotonic", "sigmoid"]:
        print(f"Calibrando {name} ({method})...", end=" ", flush=True)
        cal = CalibratedClassifierCV(pipe, cv="prefit", method=method)
        cal.fit(X_v, y_v)  # calibrar sobre val
        proba_cal = cal.predict_proba(X_te)
        preds_cal = cal.classes_[np.argmax(proba_cal, axis=1)]
        acc_cal   = accuracy_score(y_te, preds_cal)
        ece_cal   = ece(y_te, proba_cal, list(cal.classes_))
        ll_cal    = log_loss(y_te, proba_cal, labels=cal.classes_)
        calib_results[name][method] = {"proba": proba_cal, "acc": acc_cal,
                                        "ece": ece_cal, "ll": ll_cal}
        print(f"Acc={acc_cal*100:.2f}%  ECE={ece_cal:.4f}")

In [ ]:
# ── Tabla resumen ECE ─────────────────────────────────────────────────────────
rows = []
for name in models_base:
    row = {"Modelo": name,
           "ECE base": round(results[name]["ece"],4),
           "ECE isotonic": round(calib_results[name]["isotonic"]["ece"],4),
           "ECE sigmoid":  round(calib_results[name]["sigmoid"]["ece"],4),
           "Acc base (%)": round(results[name]["acc"]*100,2)}
    rows.append(row)
df_ece = pd.DataFrame(rows)
print(df_ece.to_string(index=False))

In [ ]:
# ── Reliability Diagrams (promedio OvR, top 5 clases) ────────────────────────
fig, axes = plt.subplots(len(models_base), 3, figsize=(15, 4*len(models_base)))
for row_i, (name, res) in enumerate(results.items()):
    proba_sets = {
        "Base": res["proba"],
        "Isotonic": calib_results[name]["isotonic"]["proba"],
        "Sigmoid":  calib_results[name]["sigmoid"]["proba"],
    }
    cls = res["classes"]
    y_bin = label_binarize(y_te, classes=cls)
    for col_i, (method_name, proba) in enumerate(proba_sets.items()):
        ax = axes[row_i][col_i]
        # Promediar sobre todas las clases
        frac_pos_all, mean_pred_all = [], []
        for c in range(proba.shape[1]):
            if y_bin[:, c].sum() == 0: continue
            try:
                fp, mp = calibration_curve(y_bin[:, c], proba[:, c], n_bins=10, strategy="quantile")
                frac_pos_all.append(fp); mean_pred_all.append(mp)
            except: pass
        if frac_pos_all:
            fp_mean = np.mean(frac_pos_all, axis=0)
            mp_mean = np.mean(mean_pred_all, axis=0)
            ax.plot(mp_mean, fp_mean, "s-", color="#2E75B6", markersize=5, label="Modelo")
        ax.plot([0,1],[0,1],"k--",linewidth=1,label="Perfecta")
        ece_v = calib_results[name][method_name.lower()]["ece"] if method_name != "Base" else res["ece"]
        ax.set_title(f"{name}\n{method_name}  ECE={ece_v:.4f}", fontsize=8, fontweight="bold")
        ax.set_xlabel("Confianza media",fontsize=8); ax.set_ylabel("Fracción positiva",fontsize=8)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle("Reliability Diagrams (promedio OvR)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("calibracion_reliability.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Entropía de predicción por temperatura real ───────────────────────────────
temps_real = df[df["split"]=="test"]["temperature"].values
best_name = min(results, key=lambda n: results[n]["ece"])
proba_best = results[best_name]["proba"]
entropy = -np.sum(proba_best * np.log(proba_best + 1e-12), axis=1)

temps_unique = sorted(np.unique(temps_real))
ent_by_temp = {t: entropy[temps_real == t] for t in temps_unique}

fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot([ent_by_temp[t] for t in temps_unique],
           labels=[str(int(t)) for t in temps_unique],
           patch_artist=True,
           boxprops=dict(facecolor="#BDD7EE", color="#2E75B6"),
           medianprops=dict(color="#C00000", linewidth=2))
ax.set_xlabel("Temperatura real [°C]", fontsize=10)
ax.set_ylabel("Entropía de predicción", fontsize=10)
ax.set_title(f"Incertidumbre del modelo ({best_name}) por temperatura\n"
             f"(entropía alta = modelo más inseguro)", fontsize=11, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("calibracion_entropia_por_temp.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Comparación ECE antes/después ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models_base))
w = 0.25
names = list(models_base.keys())
ax.bar(x - w, [results[n]["ece"]                            for n in names], w, label="Base",     color="#ED7D31", alpha=0.85)
ax.bar(x,     [calib_results[n]["isotonic"]["ece"]          for n in names], w, label="Isotonic", color="#2E75B6", alpha=0.85)
ax.bar(x + w, [calib_results[n]["sigmoid"]["ece"]           for n in names], w, label="Sigmoid",  color="#70AD47", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel("ECE (menor = mejor calibrado)", fontsize=10)
ax.set_title("Expected Calibration Error antes y después de calibrar", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("calibracion_ece_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()